# 📝 Resume Generator with Google Drive Integration

<div style="background-color: #f0f7fb; padding: 20px; border-radius: 10px; margin-bottom: 20px; border-left: 5px solid #3498db;">
    <h2 style="margin-top: 0; color: #3498db;">Overview</h2>
    <p>This interactive notebook streamlines your resume updates by:</p>
    <ol>
        <li>Downloading keywords from a Google Sheet</li>
        <li>Processing keywords to find the most relevant ones</li>
        <li>Updating your resume with these keywords</li>
        <li>Generating a professional PDF resume</li>
        <li>Uploading the PDF to Google Drive</li>
    </ol>
    <p><strong>One click to maintain an optimized, keyword-focused resume!</strong></p>
</div>

## 🔧 Configuration

Adjust the settings below to customize your resume generation process:

In [1]:
# Configuration Settings
config = {
    # Google Drive Integration
    "use_google_drive": True,                          # Set to False to skip Google Drive operations
    "sheet_id": "1C20hzxMQzT310HSe9DhH8XigArW5VcjAGgQfGafF4o4", # Google Sheet ID
    "folder_id": "1s6n9eS9d2NNy9lyXgoYdPGOwkyGGujnw",  # Google Drive folder ID for uploads
    
    # File Paths
    "resume_template": "templates/resume.md",          # Resume template path
    "output_filename": "resume",                       # Base name for output files (without extension)
    
    # Keyword Processing
    "top_keywords": 20,                                # Number of top keywords to include
    
    # PDF Settings
    "generate_pdf": True,                              # Try to generate PDF with pandoc
    
    # Display Settings
    "verbose_output": True,                            # Show detailed processing logs
    "show_keyword_count": 15                           # Number of keywords to display in results
}

# Display the configuration
print("Configuration loaded successfully.")

Configuration loaded successfully.


## 🚀 Generate Your Resume

Run this cell to execute the entire resume generation process using the configuration above.

In [2]:
import os
import sys
import time
import datetime
import pandas as pd
from IPython.display import HTML, display, Markdown, FileLink
import ipywidgets as widgets

# Setup paths
notebook_dir = os.path.abspath('')
base_dir = notebook_dir
scripts_dir = os.path.join(base_dir, 'scripts')

# Add scripts directory to path
sys.path.append(base_dir)
sys.path.append(scripts_dir)

# Import custom modules
import scripts.keywords_processor as keywords_processor
import scripts.update_resume as update_resume

# Check for Google Drive integration
if config["use_google_drive"]:
    try:
        import scripts.google_drive_handler as google_drive
        google_drive_available = True
    except ImportError:
        print("⚠️ Google Drive integration unavailable. Continuing without it.")
        google_drive_available = False
else:
    google_drive_available = False

# Configure file paths based on settings
input_keywords_file = os.path.join(base_dir, "data/input/lead_gen.csv")
processed_keywords_file = os.path.join(base_dir, "data/output/processed_keywords.csv")
input_resume_file = os.path.join(base_dir, config["resume_template"])
output_resume_file = os.path.join(base_dir, f"data/output/{config['output_filename']}.md")
output_pdf_file = os.path.join(base_dir, f"data/output/{config['output_filename']}.pdf")

# Create directories if they don't exist
os.makedirs(os.path.dirname(input_keywords_file), exist_ok=True)
os.makedirs(os.path.dirname(output_resume_file), exist_ok=True)

# Styled logging function
def log(message, level="INFO", display_only=False):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    if level == "INFO":
        color = "#3498db"  # Blue
        icon = "ℹ️"
    elif level == "SUCCESS":
        color = "#2ecc71"  # Green
        icon = "✅"
    elif level == "WARNING":
        color = "#f39c12"  # Orange
        icon = "⚠️"
    elif level == "ERROR":
        color = "#e74c3c"  # Red
        icon = "❌"
    else:
        color = "#7f8c8d"  # Gray
        icon = "🔹"
    
    html = f"<div style='margin-bottom: 8px;'><span style='color: {color}; font-weight: bold;'>{icon} {level}</span> <span style='color: #7f8c8d; font-size: 0.9em;'>[{timestamp}]</span> {message}</div>"
    display(HTML(html))
    
    if not display_only and config["verbose_output"]:
        print(f"[{timestamp}] {level}: {message}")

# Show a section header
def show_section(title, description=None):
    display(HTML(f"""
    <div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; margin: 20px 0 10px 0; border-left: 5px solid #6c757d;">
        <h3 style="margin: 0; color: #343a40;">{title}</h3>
        {f'<p style="margin: 5px 0 0 0; color: #6c757d;">{description}</p>' if description else ''}
    </div>
    """))

# Show a summary box 
def show_summary_box(title, items):
    html = f"""
    <div style="background-color: #f8f9fa; padding: 15px; border-radius: 5px; margin: 10px 0; border: 1px solid #dee2e6;">
        <h4 style="margin-top: 0; color: #343a40;">{title}</h4>
        <ul style="margin-bottom: 0; padding-left: 20px;">
    """
    
    for item in items:
        html += f"<li>{item}</li>"
    
    html += """
        </ul>
    </div>
    """
    
    display(HTML(html))

# Show keywords in a nice table
def show_keywords_table(df, top_n=10, title="Top Keywords"):
    # Create a copy with limited rows
    display_df = df.head(top_n).copy()
    
    # Style the dataframe
    styled_df = display_df.style.set_caption(title)\
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('color', '#343a40')]},
            {'selector': 'th', 'props': [('background-color', '#f8f9fa'), ('color', '#343a40')]},
            {'selector': 'td', 'props': [('padding', '5px 10px')]}
        ])\
        .set_properties(**{'text-align': 'left'})
    
    display(styled_df)

# Display file link with nice formatting
def show_file_link(file_path, description):
    if os.path.exists(file_path):
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path) / 1024  # KB
        size_str = f"{file_size:.1f} KB" if file_size < 1024 else f"{file_size/1024:.1f} MB"
        
        display(HTML(f"""
        <div style="margin: 10px 0; padding: 10px; border-radius: 5px; background-color: #f8f9fa; display: flex; align-items: center;">
            <div style="margin-right: 15px; font-size: 24px;">📄</div>
            <div>
                <div style="font-weight: bold;">{file_name}</div>
                <div style="color: #6c757d; font-size: 0.9em;">{description} • {size_str}</div>
            </div>
            <div style="margin-left: auto;">
                <a href="{file_path}" target="_blank" style="text-decoration: none; background-color: #007bff; color: white; padding: 5px 10px; border-radius: 3px;">View</a>
            </div>
        </div>
        """))
    else:
        log(f"File not found: {file_path}", "WARNING")

# Show a card for Google Drive links
def show_drive_link(url, title, description):
    display(HTML(f"""
    <div style="margin: 10px 0; padding: 15px; border-radius: 5px; background-color: #f0f7fb; border-left: 5px solid #3498db;">
        <div style="display: flex; align-items: center;">
            <div style="margin-right: 15px; font-size: 24px;">🔗</div>
            <div>
                <div style="font-weight: bold;">{title}</div>
                <div style="color: #6c757d; font-size: 0.9em;">{description}</div>
            </div>
            <div style="margin-left: auto;">
                <a href="{url}" target="_blank" style="text-decoration: none; background-color: #4285F4; color: white; padding: 5px 10px; border-radius: 3px;">Open in Google Drive</a>
            </div>
        </div>
    </div>
    """))

# Progress bar for long-running operations
def create_progress_bar(description="Processing"):
    bar = widgets.IntProgress(
        value=0,
        min=0,
        max=100,
        description=description,
        bar_style='info',
        style={'description_width': 'initial'}
    )
    display(bar)
    return bar

# Main execution function
def run_resume_generation():
    # Show welcome message
    display(HTML("""<h2 style="color: #3498db;">🚀 Resume Generation Process</h2>"""))
    log("Starting resume generation process...", "INFO")
    
    # === Step 1: Download from Google Drive ===
    show_section("Step 1: Retrieve Keyword Data", "Downloading data from Google Drive or using local file")
    
    if config["use_google_drive"] and google_drive_available:
        log(f"Downloading Google Sheet {config['sheet_id']} to {input_keywords_file}", "INFO")
        progress = create_progress_bar("Downloading")
        
        # Progress simulation for download (actual download doesn't report progress)
        for i in range(10):
            progress.value = i * 10
            time.sleep(0.1)
        
        success = google_drive.download_spreadsheet(config["sheet_id"], input_keywords_file)
        progress.value = 100
        
        if success:
            log("Google Sheet downloaded successfully", "SUCCESS")
            # Show a preview of the downloaded data
            try:
                keywords_df = pd.read_csv(input_keywords_file)
                display(HTML("<p><strong>Preview of downloaded keywords:</strong></p>"))
                display(keywords_df.head())
            except Exception as e:
                log(f"Error previewing downloaded file: {e}", "WARNING")
        else:
            log("Failed to download Google Sheet. Using existing file if available.", "WARNING")
    else:
        if not config["use_google_drive"]:
            log("Google Drive integration disabled in configuration. Using local file.", "INFO")
        elif not google_drive_available:
            log("Google Drive integration not available. Using local file.", "WARNING")
        
        # Check if local file exists
        if os.path.exists(input_keywords_file):
            log(f"Using existing keywords file: {input_keywords_file}", "INFO")
            try:
                keywords_df = pd.read_csv(input_keywords_file)
                display(HTML("<p><strong>Preview of local keywords file:</strong></p>"))
                display(keywords_df.head())
            except Exception as e:
                log(f"Error reading local file: {e}", "WARNING")
        else:
            log(f"Local keywords file not found: {input_keywords_file}", "ERROR")
            return
    
    # === Step 2: Process Keywords ===
    show_section("Step 2: Process Keywords", "Analyzing and extracting the most relevant keywords")
    log(f"Processing keywords from {input_keywords_file}", "INFO")
    
    try:
        progress = create_progress_bar("Processing keywords")
        progress.value = 30  # Initial progress
        
        # Process keywords
        high_df, low_df = keywords_processor.process_keywords(
            input_keywords_file, 
            processed_keywords_file
        )
        
        progress.value = 100
        log("Keywords processed successfully", "SUCCESS")
        
        # Display keyword results
        show_keywords_table(high_df, config["show_keyword_count"], "Top High-Priority Keywords")
        show_keywords_table(low_df, config["show_keyword_count"], "Top Low-Priority Keywords")
        
        show_file_link(processed_keywords_file, "Processed keywords data")
        
    except Exception as e:
        log(f"Error processing keywords: {e}", "ERROR")
        return
    
    # === Step 3: Update Resume ===
    show_section("Step 3: Update Resume", "Updating resume with processed keywords")
    log("Starting resume update process", "INFO")
    
    try:
        # Load top keywords
        keywords_df = pd.read_csv(processed_keywords_file)
        high_priority = keywords_df[(keywords_df['priority'] == 'high')].sort_values('count', ascending=False).head(config["top_keywords"])
        keywords = high_priority['keyword'].tolist()
        
        log(f"Updating resume with top {len(keywords)} keywords", "INFO")
        
        # Update resume
        progress = create_progress_bar("Updating resume")
        progress.value = 50  # Initial progress
        
        update_resume.highlight_keywords(
            input_resume_file,
            output_resume_file,
            keywords
        )
        
        progress.value = 100
        log("Resume updated successfully", "SUCCESS")
        
        # Show link to the updated markdown file
        show_file_link(output_resume_file, "Updated resume in markdown format")
        
    except Exception as e:
        log(f"Error updating resume: {e}", "ERROR")
        return
    
    # === Step 4: Generate PDF ===
    show_section("Step 4: Generate PDF", "Creating a professional PDF from the updated resume")
    
    if config["generate_pdf"]:
        log("Starting PDF generation", "INFO")
        
        try:
            progress = create_progress_bar("Generating PDF")
            progress.value = 20  # Initial progress
            
            # Generate PDF
            pdf_path = update_resume.generate_pdf(
                output_resume_file,
                os.path.dirname(output_pdf_file)
            )
            
            progress.value = 100
            
            if pdf_path and os.path.exists(pdf_path):
                log("PDF generation complete", "SUCCESS")
                show_file_link(pdf_path, "Generated resume PDF")
            else:
                # Check if PDF exists from previous runs
                if os.path.exists(output_pdf_file):
                    log("Using existing PDF file", "WARNING")
                    pdf_path = output_pdf_file
                    show_file_link(pdf_path, "Existing resume PDF")
                else:
                    log("PDF generation failed and no existing PDF found", "ERROR")
                    log("Consider using VS Code with Markdown PDF extension", "INFO")
                    pdf_path = None
        except Exception as e:
            log(f"Error during PDF generation: {e}", "ERROR")
            pdf_path = None
            
            # Check if PDF exists despite the error
            if os.path.exists(output_pdf_file):
                log(f"Using existing PDF despite error", "WARNING")
                pdf_path = output_pdf_file
                show_file_link(pdf_path, "Existing resume PDF")
    else:
        log("PDF generation skipped based on configuration", "INFO")
        pdf_path = None
    
    # === Step 5: Upload to Google Drive ===
    web_link = None
    if config["use_google_drive"] and google_drive_available and pdf_path and os.path.exists(pdf_path):
        show_section("Step 5: Upload to Google Drive", "Sharing your resume on Google Drive")
        log(f"Uploading PDF {pdf_path} to Google Drive folder {config['folder_id']}", "INFO")
        
        try:
            progress = create_progress_bar("Uploading to Google Drive")
            
            # Progress simulation for upload (actual upload doesn't report progress)
            for i in range(10):
                progress.value = i * 10
                time.sleep(0.1)
            
            web_link = google_drive.upload_pdf_to_drive(
                pdf_path, 
                config["folder_id"]
            )
            
            progress.value = 100
            
            if web_link:
                log("PDF uploaded successfully to Google Drive", "SUCCESS")
                show_drive_link(web_link, "Resume PDF on Google Drive", "Your resume has been shared on Google Drive")
            else:
                log("Failed to upload PDF to Google Drive", "WARNING")
        except Exception as e:
            log(f"Error uploading to Google Drive: {e}", "ERROR")
    elif config["use_google_drive"] and google_drive_available:
        if not pdf_path or not os.path.exists(pdf_path):
            log("Skipping Google Drive upload because PDF was not generated", "WARNING")
    else:
        if not config["use_google_drive"]:
            log("Google Drive upload skipped based on configuration", "INFO")
        elif not google_drive_available:
            log("Google Drive integration not available for upload", "WARNING")
    
    # === Final Summary ===
    show_section("✨ Process Complete", "Your resume has been generated successfully")
    
    summary_items = [
        f"<strong>Input keywords:</strong> {input_keywords_file}",
        f"<strong>Processed keywords:</strong> {processed_keywords_file}",
        f"<strong>Updated resume:</strong> {output_resume_file}"
    ]
    
    if pdf_path and os.path.exists(pdf_path):
        summary_items.append(f"<strong>Generated PDF:</strong> {pdf_path}")
    
    if web_link:
        summary_items.append(f"<strong>Google Drive link:</strong> <a href='{web_link}' target='_blank'>{web_link}</a>")
    
    show_summary_box("Resume Generation Summary", summary_items)
    
    # Show top 10 keywords used
    display(HTML("<h4 style='margin-top: 20px;'>Top 10 Keywords Used</h4>"))
    for i, keyword in enumerate(keywords[:10]):
        count = high_priority.iloc[i]['count']
        display(HTML(f"<div style='margin-left: 20px;'>{i+1}. <strong>{keyword}</strong> (Count: {count})</div>"))
    
    log("All processing completed successfully!", "SUCCESS")

# Run the resume generation process
run_resume_generation()

⚠️ Google Drive integration unavailable. Continuing without it.


[2025-03-20 19:04:34] INFO: Starting resume generation process...


[2025-03-20 19:04:34] WARNING: Google Drive integration not available. Using local file.


[2025-03-20 19:04:34] INFO: Using existing keywords file: /Users/malachi/Documents/Github/keywords/data/input/lead_gen.csv


,high_priority_keywords,low_priority_keywords
0,Looker,interpersonal skills
1,Python,attention to detail
2,SQL,problem-solving
3,back-end,decision-making
4,data solutions,decision-making


...

IntProgress(value=0, bar_style='info', description='Generating PDF', style=ProgressStyle(description_width='in…

Using pandoc at: pandoc
PDF successfully generated: /Users/malachi/Documents/Github/keywords/data/output/resume.pdf


[2025-03-20 19:04:38] SUCCESS: PDF generation complete


[2025-03-20 19:04:38] WARNING: Google Drive integration not available for upload


[2025-03-20 19:04:38] SUCCESS: All processing completed successfully!


## 📊 Advanced Options

The cell below allows you to use the individual components separately for more control over the process.

In [ ]:
# This cell is optional - only run if you want to perform individual operations

import ipywidgets as widgets
from IPython.display import display, HTML

def on_operation_selected(change):
    global operation_output
    operation_output.clear_output()
    
    with operation_output:
        if change.new == "download":
            sheet_id = widgets.Text(
                value=config["sheet_id"],
                description="Sheet ID:",
                style={'description_width': 'initial'}
            )
            output_path = widgets.Text(
                value=input_keywords_file,
                description="Output file:",
                style={'description_width': 'initial'}
            )
            download_button = widgets.Button(description="Download")
            display(HTML("<h3>Download from Google Sheet</h3>"))
            display(HTML("<p>Enter the Google Sheet ID and output file path:</p>"))
            display(sheet_id)
            display(output_path)
            display(download_button)
            
            def on_download_clicked(b):
                download_output.clear_output()
                with download_output:
                    try:
                        log(f"Downloading from Sheet ID: {sheet_id.value}", "INFO")
                        success = google_drive.download_spreadsheet(sheet_id.value, output_path.value)
                        if success:
                            log("Download successful!", "SUCCESS")
                            show_file_link(output_path.value, "Downloaded keywords file")
                        else:
                            log("Download failed.", "ERROR")
                    except Exception as e:
                        log(f"Error during download: {e}", "ERROR")
            
            download_button.on_click(on_download_clicked)
            download_output = widgets.Output()
            display(download_output)
            
        elif change.new == "process":
            input_path = widgets.Text(
                value=input_keywords_file,
                description="Input file:",
                style={'description_width': 'initial'}
            )
            output_path = widgets.Text(
                value=processed_keywords_file,
                description="Output file:",
                style={'description_width': 'initial'}
            )
            process_button = widgets.Button(description="Process Keywords")
            display(HTML("<h3>Process Keywords</h3>"))
            display(HTML("<p>Enter the input and output file paths:</p>"))
            display(input_path)
            display(output_path)
            display(process_button)
            
            def on_process_clicked(b):
                process_output.clear_output()
                with process_output:
                    try:
                        log(f"Processing keywords from {input_path.value}", "INFO")
                        high_df, low_df = keywords_processor.process_keywords(input_path.value, output_path.value)
                        log("Keywords processed successfully", "SUCCESS")
                        show_keywords_table(high_df, 10, "Top High-Priority Keywords")
                        show_keywords_table(low_df, 10, "Top Low-Priority Keywords")
                        show_file_link(output_path.value, "Processed keywords file")
                    except Exception as e:
                        log(f"Error processing keywords: {e}", "ERROR")
            
            process_button.on_click(on_process_clicked)
            process_output = widgets.Output()
            display(process_output)
            
        elif change.new == "update":
            resume_path = widgets.Text(
                value=input_resume_file,
                description="Resume template:",
                style={'description_width': 'initial'}
            )
            keywords_path = widgets.Text(
                value=processed_keywords_file,
                description="Keywords file:",
                style={'description_width': 'initial'}
            )
            output_path = widgets.Text(
                value=output_resume_file,
                description="Output file:",
                style={'description_width': 'initial'}
            )
            update_button = widgets.Button(description="Update Resume")
            display(HTML("<h3>Update Resume</h3>"))
            display(HTML("<p>Enter the template, keywords, and output file paths:</p>"))
            display(resume_path)
            display(keywords_path)
            display(output_path)
            display(update_button)
            
            def on_update_clicked(b):
                update_output.clear_output()
                with update_output:
                    try:
                        log("Loading keywords...", "INFO")
                        keywords_df = pd.read_csv(keywords_path.value)
                        high_priority = keywords_df[(keywords_df['priority'] == 'high')].sort_values('count', ascending=False).head(config["top_keywords"])
                        keywords = high_priority['keyword'].tolist()
                        
                        log(f"Updating resume with top {len(keywords)} keywords", "INFO")
                        update_resume.highlight_keywords(resume_path.value, output_path.value, keywords)
                        log("Resume updated successfully", "SUCCESS")
                        show_file_link(output_path.value, "Updated resume file")
                    except Exception as e:
                        log(f"Error updating resume: {e}", "ERROR")
            
            update_button.on_click(on_update_clicked)
            update_output = widgets.Output()
            display(update_output)
            
        elif change.new == "pdf":
            input_path = widgets.Text(
                value=output_resume_file,
                description="Markdown file:",
                style={'description_width': 'initial'}
            )
            output_dir = widgets.Text(
                value=os.path.dirname(output_pdf_file),
                description="Output directory:",
                style={'description_width': 'initial'}
            )
            pdf_button = widgets.Button(description="Generate PDF")
            display(HTML("<h3>Generate PDF</h3>"))
            display(HTML("<p>Enter the markdown file and output directory:</p>"))
            display(input_path)
            display(output_dir)
            display(pdf_button)
            
            def on_pdf_clicked(b):
                pdf_output.clear_output()
                with pdf_output:
                    try:
                        log(f"Generating PDF from {input_path.value}", "INFO")
                        pdf_path = update_resume.generate_pdf(input_path.value, output_dir.value)
                        if pdf_path and os.path.exists(pdf_path):
                            log("PDF generated successfully", "SUCCESS")
                            show_file_link(pdf_path, "Generated PDF file")
                        else:
                            log("PDF generation failed", "ERROR")
                    except Exception as e:
                        log(f"Error generating PDF: {e}", "ERROR")
            
            pdf_button.on_click(on_pdf_clicked)
            pdf_output = widgets.Output()
            display(pdf_output)
            
        elif change.new == "upload":
            pdf_path = widgets.Text(
                value=output_pdf_file,
                description="PDF file:",
                style={'description_width': 'initial'}
            )
            folder_id = widgets.Text(
                value=config["folder_id"],
                description="Folder ID:",
                style={'description_width': 'initial'}
            )
            upload_button = widgets.Button(description="Upload to Drive")
            display(HTML("<h3>Upload to Google Drive</h3>"))
            display(HTML("<p>Enter the PDF file and Google Drive folder ID:</p>"))
            display(pdf_path)
            display(folder_id)
            display(upload_button)
            
            def on_upload_clicked(b):
                upload_output.clear_output()
                with upload_output:
                    try:
                        log(f"Uploading {pdf_path.value} to folder {folder_id.value}", "INFO")
                        web_link = google_drive.upload_pdf_to_drive(pdf_path.value, folder_id.value)
                        if web_link:
                            log("Upload successful!", "SUCCESS")
                            show_drive_link(web_link, "Resume on Google Drive", "Your resume has been uploaded to Google Drive")
                        else:
                            log("Upload failed", "ERROR")
                    except Exception as e:
                        log(f"Error uploading to Google Drive: {e}", "ERROR")
            
            upload_button.on_click(on_upload_clicked)
            upload_output = widgets.Output()
            display(upload_output)

# Create operation selection dropdown
operation = widgets.Dropdown(
    options=[
        ('Select an operation', 'none'),
        ('Download from Google Sheet', 'download'),
        ('Process Keywords', 'process'),
        ('Update Resume', 'update'),
        ('Generate PDF', 'pdf'),
        ('Upload to Google Drive', 'upload')
    ],
    value='none',
    description='Operation:',
    style={'description_width': 'initial'}
)

operation.observe(on_operation_selected, names='value')
display(HTML("<h3>Advanced Operations</h3>"))
display(HTML("<p>Select an individual operation to perform:</p>"))
display(operation)
operation_output = widgets.Output()
display(operation_output)

Dropdown(description='Operation:', options=(('Select an operation', 'none'), ('Download from Google Sheet', 'd…

Output()

## 📋 Project Information

<div style="background-color: #f0f0f0; padding: 15px; border-radius: 5px;">
    <h3 style="margin-top: 0;">About the Resume Generator</h3>
    <p>This tool helps you maintain a keyword-optimized resume by analyzing job descriptions, extracting relevant keywords, and updating your resume to highlight your skills that match these keywords. The process is fully automated from downloading data to uploading the final PDF.</p>
    
    <h4>Key Features</h4>
    <ul>
        <li><strong>Google Drive Integration:</strong> Download keyword data directly from Google Sheets and upload the generated PDF back to Google Drive</li>
        <li><strong>Smart Keyword Processing:</strong> Preserves multi-word terms like "machine learning" and prioritizes keywords based on frequency</li>
        <li><strong>PDF Generation:</strong> Creates a professional PDF resume using pandoc with proper formatting</li>
        <li><strong>Interactive Interface:</strong> Run the entire process with a single click, with visual feedback and progress indicators</li>
    </ul>
</div>

<div style="margin-top: 20px; text-align: center; color: #777; font-size: 0.9em;">
    <p>Created by Malachi Dunn • © 2024 • <a href="https://github.com/malachi-mindwyre/resume">GitHub Repository</a></p>
</div>